# Everyone's dashboard is green. The system is broken.

**Paper:** *Projection Obstruction for Factorized States* (PO-000, v6.3)

`AuthorityEffect: None.` Nothing here authorises any decision.

---

## The idea, with no mathematics

Picture a system split into parts, with a monitor watching each part. Each
monitor reports how much is wrong in *its* part. Every monitor reports zero.

The natural conclusion is that nothing is wrong. This notebook shows a case
where that conclusion is **false**: every monitor honestly reports zero while
the total amount wrong is six.

The monitors are not broken, biased, or lying. Each is a faithful measurement
of what it can see. The problem is *where* the fault lies: each monitor sees only a projection of its part, and the fault sits in a direction that monitor's projection cannot see. It is not between the monitors — it is inside one, in a direction it is blind to.

An analogy: three people each audit one department's books, and each
department balances. But money moved between departments in a way no single
audit covers, so the company's books do not balance. Adding more honest
auditors does not help if none of them looks across the boundary.

## Why prove this rather than just assert it

"Local checks can miss global problems" is folklore. What the paper does is
make it exact: it builds a specific finite example, computes both numbers, and
shows the gap is not a rounding artefact or a badly chosen monitor. The gap is
**exactly 6** and every observed total is **exactly 0**, in exact rational
arithmetic with no floating point anywhere.

It then shows the gap can be made **as large as you like** while every monitor
still reads zero — so "the local readings were small, so the global problem
must be small" is never safe reasoning here.

## Three words, then we compute

| Paper's symbol | Plain meaning |
| --- | --- |
| **I(s)** | the true total amount wrong, across the whole system |
| **J^α(s)** | what monitor α reports: the part it can actually see |
| **Λ** | the gap — how much the best single monitor misses |

"All-Green" means every J is zero. "Global Red" means I is not zero.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str((Path.cwd() if (Path.cwd() / "code").exists() else Path.cwd().parent).resolve() / "code"))

from observer_carriers import scaling_family_certificate, three_patch_certificate

print("loaded observer_carriers from code/")

## Exhibit 1 — the three-patch system

Three monitors, full coverage: every part is watched by at least one of them.
Run it and compare the two numbers.

In [ ]:
cert = three_patch_certificate()

print("Is every part of the system watched by someone?", cert.full_edge_coverage)
print()
print("What each monitor reports (J):")
for observer, reported in cert.tallies.J.items():
    print(f"    monitor {observer}: {reported}")
print()
print(f"Every dashboard green?           {all(v == 0 for v in cert.tallies.J.values())}")
print(f"True total amount wrong  I(s):   {cert.tallies.I}")
print(f"Gap the monitors miss    Lambda: {cert.tallies.Lambda}")

Every monitor reads **0**. The truth is **6**.

These are exact integers, not approximations — there is no tolerance to argue
about and no floating-point noise. The deposit's arithmetic runs on exact
fractions throughout, so `6` means six.

## Exhibit 2 — the gap has no ceiling

A natural hope: maybe the miss is bounded, so green dashboards at least mean
*small* trouble. Scaling the same system refutes that.

In [ ]:
scales = [1, 2, 10, 100, 1000]
family = scaling_family_certificate(scales)

print("scale     true total wrong     worst monitor reading")
print("-" * 54)
for scale, instance in zip(scales, family.instances):
    worst = max(instance.tallies.J.values())
    # str() first: Fraction.__format__ only exists from CPython 3.12, and
    # pyproject declares requires-python >= 3.11, where a format spec on a
    # Fraction raises TypeError.
    print(f"{scale:>5}     {str(instance.tallies.I):>15}     {str(worst):>21}")
print()
print("Scaling is exactly linear:", family.exact_linear_scaling)

The true total grows without limit. Every monitor still reads zero at every
scale.

So a green dashboard bounds nothing at all. This is the paper's Corollary
7.4.6: for this family there is **no recovery function** — no rule of the form
"if all local readings are below *x*, the global problem is below *f(x)*".

## So when *does* all-green mean all-clear?

The paper answers this, and the answer is sharp in both directions.

**Sufficient:** if the monitors on each part together have nothing in their
blind spot, all-green really does force zero (Corollary 7.4.7).

**Not necessary:** the paper also builds a system with genuine blind spots
everywhere where all-green *still* forces zero, because the parts are coupled
(Proposition 7.4.7-B). So "my monitors have blind spots" does not by itself
mean you are exposed — it depends on the coupling.

That second half matters: it stops the first half from being read as a scare
story. The honest summary is that blind spots are the thing to reason about,
not something to panic about.

## What this does not say

Stated plainly, because the paper is careful about this and the notebook should
be too:

- It is **not** a claim about any real monitoring system, organisation, or
  physical experiment. It is an exact statement about a finite mathematical
  object.
- It does **not** say local checks are useless. It says they do not *bound* the
  global quantity without an extra condition — and it states that condition.
- Running this notebook is evidence about these displayed instances only. It is
  not a proof of the general propositions; those are proved in the paper and
  machine-checked in Lean 4 under `lean4/`.

The claim ledger records exactly which parts are proved, which are mechanised,
and which remain open. Nothing here authorises a decision.